Goal is to clean up current Kalshi positions to free up capital for new trades. The positions are currently in a state where they are not profitable and are taking up capital that could be used for other trades. The goal is to close these positions and free up capital for new trades.

In [1]:
import polars as pl
import os
import sys
from datetime import datetime, timezone

# Ensure project root is on path and cwd so relative key paths resolve
PROJECT_ROOT = os.path.dirname(os.path.abspath("."))
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from dotenv import load_dotenv
from rl_bot.kalshi_api import KalshiRESTClient
import config

load_dotenv()

# Initialize API client with RSA key file path
client = KalshiRESTClient(
    api_key=config.API_KEY,
    api_secret=config.KEY_PATH
)

print("Client initialized")

Client initialized


In [2]:
# ── Cell 2: Pull all positions and enrich with market data ──

response = client.get_positions()
positions = response.get("positions", [])

# Filter to positions with actual holdings
open_positions = [p for p in positions if p.get("total_traded", 0) != 0]
print(f"Open positions: {len(open_positions)}")

# Build raw dataframe from position data
pos_df = pl.DataFrame(open_positions)
print(pos_df.columns)
pos_df.head(5)

Open positions: 0
[]


shape: (0, 0)
┌┐
╞╡
└┘

In [3]:
# ── Cell 3: Enrich each position with market details + orderbook ──
# Fetch market info and best bid/ask for every open position

enriched = []
now_ts = int(datetime.now(timezone.utc).timestamp())

for pos in open_positions:
    ticker = pos.get("market_ticker", "")
    net_pos = pos.get("total_traded", 0)  # positive = long YES, negative = long NO

    # Pull market metadata (close_time, status)
    try:
        mkt = client.get_market(ticker).get("market", {})
    except Exception:
        mkt = {}

    # Pull top-of-book
    try:
        ob = client.get_orderbook(ticker, depth=1)
        best_bid = ob.get("orderbook", {}).get("yes", [[None]])[0][0]  # best YES bid price
        best_ask = ob.get("orderbook", {}).get("no", [[None]])[0][0]   # best NO bid = YES ask
        # Normalize to cents if needed
        if best_bid and best_bid > 1:
            best_bid = best_bid / 100.0
        if best_ask and best_ask > 1:
            best_ask = 1.0 - best_ask / 100.0  # NO bid -> YES ask
    except Exception:
        best_bid, best_ask = None, None

    # Parse close time
    close_time_str = mkt.get("close_time", "")
    try:
        close_dt = datetime.fromisoformat(close_time_str.replace("Z", "+00:00"))
        hours_to_close = (close_dt.timestamp() - now_ts) / 3600.0
    except Exception:
        close_dt = None
        hours_to_close = None

    enriched.append({
        "ticker": ticker,
        "category": ticker.split("-")[0] if "-" in ticker else "UNK",
        "net_position": net_pos,
        "direction": "LONG" if net_pos > 0 else "SHORT",
        "contracts": abs(net_pos),
        "best_bid": best_bid,
        "best_ask": best_ask,
        "mid_price": (best_bid + best_ask) / 2 if best_bid and best_ask else None,
        "spread": (best_ask - best_bid) if best_bid and best_ask else None,
        "status": mkt.get("status", "unknown"),
        "close_time": close_time_str,
        "hours_to_close": hours_to_close,
    })

df = pl.DataFrame(enriched)
print(f"Enriched {len(df)} positions")
df.head(10)

Enriched 0 positions


shape: (0, 0)
┌┐
╞╡
└┘

In [4]:
# ── Cell 4: Liquidation scoring framework ──
# Score each position on how urgently it should be liquidated.
# Higher score = liquidate first.
#
# Factors:
#   1. exit_value: what you'd get if you market-sold right now (bid for longs, 1-ask for shorts)
#   2. spread_pct: wide spreads = illiquid = harder to exit = penalize
#   3. time_pressure: positions near expiry with no edge should be closed
#   4. capital_tied: contracts * mid_price = capital locked up

df_scored = df.with_columns([
    # For LONG positions, exit at best_bid; for SHORT, exit at (1 - best_ask)
    pl.when(pl.col("direction") == "LONG")
      .then(pl.col("best_bid"))
      .otherwise(1.0 - pl.col("best_ask"))
      .alias("exit_price"),
]).with_columns([
    # Capital locked up in this position (in dollars, 1 contract = $1 notional)
    (pl.col("contracts") * pl.col("mid_price")).alias("capital_tied"),

    # Spread as pct of mid — wider = less liquid = harder exit
    (pl.col("spread") / pl.col("mid_price")).alias("spread_pct"),

    # Time pressure: exponential urgency as expiry approaches
    # 0 hours = score 1.0, 24h = ~0.37, 168h (1wk) = ~0.001
    pl.when(pl.col("hours_to_close").is_not_null())
      .then((-pl.col("hours_to_close") / 24.0).exp())
      .otherwise(pl.lit(0.0))
      .alias("time_pressure"),
]).with_columns([
    # Positions near 50c (max uncertainty) tie up the most risk-capital
    # Positions near 0 or 1 are nearly resolved — less urgent to exit
    (0.5 - (pl.col("mid_price") - 0.5).abs()).alias("uncertainty_score"),
]).with_columns([
    # ── Composite liquidation score ──
    # Weights: time_pressure matters most, then capital tied, then uncertainty
    (
        0.40 * pl.col("time_pressure")
      + 0.25 * (pl.col("capital_tied") / pl.col("capital_tied").max())  # normalize
      + 0.20 * pl.col("uncertainty_score")
      + 0.15 * pl.col("spread_pct").fill_null(1.0).clip(0.0, 1.0)
    ).alias("liquidation_score"),
])

# Sort by liquidation score descending — top = sell first
df_scored = df_scored.sort("liquidation_score", descending=True)
print("LIQUIDATION PRIORITY (higher score = sell first)")
print("=" * 80)
df_scored.select([
    "ticker", "direction", "contracts", "mid_price", "spread",
    "hours_to_close", "capital_tied", "liquidation_score"
])

ColumnNotFoundError: unable to find column "direction"; valid columns: []

In [ ]:
# ── Cell 5: Summary stats + category breakdown ──

print("PORTFOLIO SUMMARY")
print("=" * 80)
total_capital = df_scored["capital_tied"].sum()
print(f"Total capital tied up: ${total_capital:.2f}")
print(f"Total positions: {len(df_scored)}")
print(f"Long: {df_scored.filter(pl.col('direction') == 'LONG').height}")
print(f"Short: {df_scored.filter(pl.col('direction') == 'SHORT').height}")
print()

# Category breakdown — which categories are eating the most capital?
cat_summary = (
    df_scored
    .group_by("category")
    .agg([
        pl.col("contracts").sum().alias("total_contracts"),
        pl.col("capital_tied").sum().alias("total_capital"),
        pl.col("liquidation_score").mean().alias("avg_liq_score"),
        pl.len().alias("num_positions"),
    ])
    .sort("total_capital", descending=True)
)
print("CAPITAL BY CATEGORY:")
print("-" * 80)
cat_summary

In [ ]:
# ── Cell 6: Actionable liquidation list ──
# Filter to positions worth liquidating: score above threshold, or expiring soon

SCORE_THRESHOLD = 0.3  # adjust after inspecting scores

to_liquidate = df_scored.filter(pl.col("liquidation_score") >= SCORE_THRESHOLD)

print(f"POSITIONS TO LIQUIDATE (score >= {SCORE_THRESHOLD}): {len(to_liquidate)}")
print(f"Capital that would be freed: ${to_liquidate['capital_tied'].sum():.2f}")
print("=" * 80)

# Show the full liquidation list with exit prices
to_liquidate.select([
    "ticker", "direction", "contracts", "exit_price",
    "mid_price", "hours_to_close", "liquidation_score"
])

In [ ]:
# ── Cell 7: Execute liquidation (DRY RUN by default) ──
# Set DRY_RUN = False to actually place orders

DRY_RUN = True

for row in to_liquidate.iter_rows(named=True):
    ticker = row["ticker"]
    direction = row["direction"]
    contracts = row["contracts"]
    exit_price = row["exit_price"]

    if exit_price is None:
        print(f"SKIP {ticker} — no exit price available")
        continue

    # To close a LONG YES position: sell YES = place ask (sell) order
    # To close a SHORT (long NO) position: sell NO = place bid (buy YES) order
    if direction == "LONG":
        side = "ask"
        price = exit_price  # sell at best bid
    else:
        side = "bid"
        price = 1.0 - exit_price  # buy YES to close NO position

    # Price in dollars for create_order
    price_dollars = round(price, 2)

    if DRY_RUN:
        print(f"[DRY RUN] {ticker}: {side} {contracts} @ ${price_dollars:.2f}")
    else:
        try:
            resp = client.create_order(
                ticker=ticker,
                side=side,
                price_dollars=price_dollars,
                count=contracts,
            )
            print(f"PLACED  {ticker}: {side} {contracts} @ ${price_dollars:.2f} -> {resp.get('order', {}).get('order_id', 'N/A')}")
        except Exception as e:
            print(f"FAILED  {ticker}: {e}")